In [ ]:
import re
import pandas as pd


# TODO: aviser si vire id_orateur et utiliser id_acteur partout
df = pd.read_csv(
    "../data/interim/data_cleaning.csv", low_memory=False, dtype={"ID_orateur": str}
)


def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return texte
    # Supprimer les balises HTML/XML
    texte = re.sub(r"<[^>]+>", "", texte)
    # Supprimer contenu entre parenthèses
    texte = re.sub(r"\([^)]*\)", "", texte)
    # Supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # uniformise pour avoir les bons apostrophes (nécessaire pour regex)
    texte = texte.replace("’", "'")
    return texte


df["Texte_clean"] = df["texte"].apply(nettoyer_texte)
df = df.dropna(subset=["Texte_clean"])

# Basique

In [ ]:
# # Exemple de regex d'exclusion Basique
# pattern_excl_case_sensitive = re.compile(r"\b[LlDd]es Républicains\b")
# pattern_excl_case_insensitive = re.compile(
#     r"\brépublique en marche\b|\bgauche démocrate et républicaine\b", re.I
# )

# pattern_lexical = re.compile(r"républi", re.I)

# =============================================
# Mais on part de la notre qui casse la tête
# =============================================

# préparer les pays à exclure
with open("../data/raw/liste_pays_republique_stable.txt", "r", encoding="utf-8") as f:
    liste_pays = [line.strip() for line in f]

# créer un pattern regex pour les pays
# ici pas besoin d'avoir un groupe de capture par pays mais juste global ok
pattern_pays = r"(\b(?:" + r"|".join(re.escape(p) for p in liste_pays) + r")\b)"


# Regex du champ lexical République (simplifié ici)

pattern_lexical = re.compile(
    r"républi",  # même au milieu des mots
    re.I,
)

# Regex des expressions à exclure

# TODO / REMINDER :
# -> faudra que nos exclusions on pense aux virgules etc.
# Cf dans ce que j’avais renvoyé à la va vite dans les contextes elles sont supprimées
# (Pas dans le nettoyage à proprement parler mais dans la tokenisation que je fais à l’arrache)
# i.e. :  motif \b[\w-]+\b -> garde surtout lettres/chiffres/_/tiret, pas la virgule

# Expressions à exclure - casse exacte
pattern_excl_case_sensitive = re.compile(
    r"\b[LlDd]es Républicains\b"  # garde la casse pour identifier le parti (et pas un adjectif)
)  # voir pour élu Républicain ? doute

# Expressions à exclure - ignorer la casse
pattern_excl_case_insensitive = re.compile(
    # partis et groupes politiques
    r"|(\bgauche démocrate et républicaine)"
    r"|(\brépublique en marche\b)"
    r"|(\bsocialiste, écologiste et républicain\b)"
    # fonctions et institutions
    r"|(\bprésident[s]? de la République\b)"
    r"|(\bprésidence[s]? de la République\b)"
    r"|(\bprocureur[s]? de la République\b)"
    r"|(\bcour[s]? de justice de la République\b)"
    r"|(\bcour[s]? de sûreté de la République\b)"
    r"|(\badministration générale de la République\b)"
    r"|(\bGouvernement de la République française\b)"
    # pays
    r"|(\brépublique[s]? soviétique[s]?\b)"  # pas un pays mais des expressions
    r"|(" + pattern_pays + ")",  # ajout des exclusions de pays si existe
    re.I,
)

# ================================================
# On garde pour s'en rappeler mais on utilise
# la version pour l'extraction de contexte dessous
# d'ailleurs pourrait améliorer avec compréhension
# de liste etc.
# ================================================


def contains_lexical_outside_excl(text):
    # Trouver les positions des expressions exclues
    excl_positions = []

    # Ajouter les exclusions sensibles à la casse
    excl_positions.extend(
        [m.span() for m in pattern_excl_case_sensitive.finditer(text)]
    )

    # Ajouter les exclusions insensibles à la casse
    excl_positions.extend(
        [m.span() for m in pattern_excl_case_insensitive.finditer(text)]
    )

    # Fonction pour vérifier si une position est dans une zone exclue
    def in_excl(pos):
        for start, end in excl_positions:
            if start <= pos < end:
                return True
        return False

    # Chercher toutes les occurences du champ lexical
    for match in pattern_lexical.finditer(text):
        start_pos = match.start()
        if not in_excl(start_pos):
            return True
    return False


# ================================================
# Version extraction ici.
# Sans doute plus propre côté syntaxe d'ailleurs
# ================================================


def extract_contexts_simple(series, keyword="républi", window=5):
    contexts = []

    for text in series.dropna():
        # Positions des exclusions
        excl_positions = []
        excl_positions.extend(
            [m.span() for m in pattern_excl_case_sensitive.finditer(text)]
        )
        excl_positions.extend(
            [m.span() for m in pattern_excl_case_insensitive.finditer(text)]
        )

        def in_excl(pos):
            return any(start <= pos < end for start, end in excl_positions)

        tokens = re.findall(r"\b[\w-]+\b", text)

        # calculer position caractère pour chaque token
        char_index = 0
        token_positions = []
        for tok in tokens:
            pos = text.find(tok, char_index)
            token_positions.append((tok, pos))
            char_index = pos + len(tok)

        # parcourir tokens pour trouver "républi"
        for i, (tok, pos) in enumerate(token_positions):
            if pattern_lexical.search(tok) and not in_excl(pos):
                start = max(0, i - window)
                end = min(len(tokens), i + window + 1)
                context = " ".join(tokens[start:end])
                contexts.append(context)

    return contexts

In [48]:
contexts_5 = extract_contexts_simple(df["Texte_clean"], window=5)
context_counts_5 = pd.Series(contexts_5).value_counts()

In [ ]:
contexts_2 = extract_contexts_simple(df["Texte_clean"], window=2)
context_counts_2 = pd.Series(contexts_2).value_counts()

In [ ]:
contexts_1 = extract_contexts_simple(df["Texte_clean"], window=1)
context_counts_1 = pd.Series(contexts_1).value_counts()

In [49]:
print("Length of context_counts_5 (window=5):", len(context_counts_5))
print("Length of context_counts_2 (window=2):", len(context_counts_2))
print("Length of context_counts_1 (window=1):", len(context_counts_1))


Length of context_counts_5 (window=5): 19370
Length of context_counts_2 (window=2): 14941
Length of context_counts_1 (window=1): 6229


In [50]:
context_counts_5.to_csv("context_counts_window_5.csv")
context_counts_2.to_csv("context_counts_window_2.csv")
context_counts_1.to_csv("context_counts_window_1.csv")